# Classification Métier des Variables ResStock
**Objectif :** Classer les 165 variables `in.*` et les 30 usages `out.electricity.*` en catégories énergétiques métier.  
**Source :** ResStock 2025 Release 1 — NREL  
**Données :** `metadata_clean.parquet` + `upgrade0.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW       = ROOT / 'data' / 'raw'
FIGURES        = ROOT / 'reports' / 'figures'

df_meta  = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')
df_raw   = pd.read_parquet(DATA_RAW / 'upgrade0.parquet')

print(f'metadata_clean : {df_meta.shape}')
print(f'upgrade0       : {df_raw.shape}')

---
## 1. Variables Socio-Économiques
Variables décrivant la situation financière et le statut social du ménage.

In [ ]:
socio = [
    ('in.income',                    'Revenu annuel du ménage (tranches $)',                   'Revenu'),
    ('in.representative_income',     'Revenu représentatif pondéré par le poids statistique', 'Revenu'),
    ('in.area_median_income',        'Revenu médian de la zone géographique (%)',              'Revenu relatif'),
    ('in.state_metro_median_income', 'Revenu médian état / métropole ($)',                     'Revenu relatif'),
    ('in.federal_poverty_level',     'Niveau de pauvreté fédéral (% du seuil FPL)',            'Pauvreté'),
    ('in.tenure',                    'Statut d\'occupation : Owner / Renter / Not Available',  'Statut logement'),
    ('in.household_has_tribal_persons', 'Présence de personnes issues de communautés tribales', 'Statut social'),
    ('in.aiannh_area',               'Localisation en zone tribale amérindienne (Yes/No)',     'Statut social'),
    ('in.vacancy_status',            'Logement occupé vs vacant',                              'Occupation'),
    ('in.units_represented',         'Nombre d\'unités représentées dans la pondération',      'Pondération'),
]

df_socio = pd.DataFrame(socio, columns=['Variable', 'Description', 'Catégorie'])
print('=== Variables Socio-Économiques ===\n')
print(df_socio.to_string(index=False))

# Aperçu des distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Revenu
income = pd.to_numeric(df_meta['in.income'], errors='coerce').dropna()
axes[0].hist(income, bins=30, color='#3b82f6', edgecolor='white')
axes[0].axvline(income.median(), color='red', linestyle='--', label=f'Médiane: ${income.median():,.0f}')
axes[0].set_title('Distribution du revenu annuel', fontweight='bold')
axes[0].set_xlabel('Revenu ($)')
axes[0].legend()

# Tenure
tenure = df_meta['in.tenure'].value_counts()
axes[1].bar(tenure.index, tenure.values, color=['#22c55e','#f59e0b','#94a3b8'])
for bar, val in zip(axes[1].patches, tenure.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2000,
                 f'{val/len(df_meta)*100:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Statut d\'occupation', fontweight='bold')

# Federal Poverty Level
fpl_vals = df_meta['in.federal_poverty_level'].value_counts().sort_index().head(15)
axes[2].barh(fpl_vals.index.astype(str), fpl_vals.values, color='#f97316', alpha=0.8)
axes[2].set_title('Niveau de pauvreté fédéral (FPL)', fontweight='bold')
axes[2].set_xlabel('Nb bâtiments')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Variables Socio-Économiques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'socio_eco_variables.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Variables Occupants
Variables décrivant les occupants et leur mode de présence.  
> **Importance énergétique :** Chaque occupant génère ~80W d'apports internes de chaleur (métabolisme).  
> Sur 549 971 bâtiments : moyenne = 2 occupants → ~160W d'apports constants, soit ~1 400 kWh/an d'effet sur le chauffage.

In [ ]:
occupants = [
    ('in.occupants',       'Nombre d\'occupants dans le logement (0–9)',                    'Haute — apports internes chaleur'),
    ('in.bedrooms',        'Nombre de chambres — proxy de la taille du foyer',              'Moyenne — corrélé au nb occupants'),
    ('in.vacancy_status',  'Logement occupé (Occupied) ou vacant (Vacant)',                 'Haute — 0 occupant = 0 apport interne'),
    ('in.usage_level',     'Niveau global d\'utilisation : Low / Medium / High',            'Haute — multiplie tous les usages'),
    ('in.tenure',          'Propriétaire vs Locataire (influence comportement et rénovation)', 'Moyenne — comportement d\'usage'),
]

df_occ = pd.DataFrame(occupants, columns=['Variable', 'Description', 'Importance énergétique'])
print('=== Variables Occupants ===\n')
print(df_occ.to_string(index=False))
print()
print('Note : ResStock ne contient pas de variables de schedule horaire (télétravail,')
print('présence/absence heure par heure). Ces données sont simulées en interne par EnergyPlus')
print('selon le usage_level (Low/Medium/High) mais non exportées dans metadata.')

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

occ = df_meta['in.occupants'].value_counts().sort_index()
axes[0].bar(occ.index.astype(str), occ.values, color='#6366f1', alpha=0.85)
axes[0].set_title('Distribution nb occupants', fontweight='bold')
axes[0].set_xlabel('Nb occupants')
axes[0].set_ylabel('Nb bâtiments')

usage = df_meta['in.usage_level'].value_counts()
colors_u = ['#22c55e', '#f59e0b', '#ef4444']
axes[1].bar(usage.index, usage.values, color=colors_u, alpha=0.85)
for bar, val in zip(axes[1].patches, usage.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2000,
                 f'{val/len(df_meta)*100:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Niveau d\'utilisation (usage_level)', fontweight='bold')

vacancy = df_meta['in.vacancy_status'].value_counts()
axes[2].pie(vacancy.values, labels=vacancy.index,
            colors=['#22c55e','#94a3b8'], autopct='%1.1f%%', startangle=90)
axes[2].set_title('Statut occupation (vacancy)', fontweight='bold')

for ax in axes[:2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Variables Occupants', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'occupants_variables.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Variables Météo / Conditions Extérieures
> **Important :** ResStock ne contient pas de données météo directes (température, humidité, vent) dans `metadata`.  
> Les simulations EnergyPlus utilisent des fichiers TMY (Typical Meteorological Year) référencés par `in.weather_file_city`.  
> Les **zones climatiques** sont les proxys météo disponibles dans le dataset.

In [ ]:
meteo = [
    ('in.weather_file_city',                    'Ville de référence du fichier météo TMY utilisé en simulation',  'Référence géo'),
    ('in.weather_file_latitude',                'Latitude du fichier météo (°N)',                                 'Géolocalisation'),
    ('in.weather_file_longitude',               'Longitude du fichier météo (°W)',                                'Géolocalisation'),
    ('in.ashrae_iecc_climate_zone_2004',         'Zone climatique ASHRAE IECC 2004 (1A–7, 8) — principale ref.', 'Zone climatique'),
    ('in.ashrae_iecc_climate_zone_2004_sub_cz_split', 'Sous-zone climatique (A=humide, B=sec, C=marin)',          'Zone climatique'),
    ('in.building_america_climate_zone',         'Zone climatique DOE Building America (Hot-Humid, Mixed-Dry…)', 'Zone climatique'),
    ('in.cec_climate_zone',                     'Zone climatique CEC (Californie uniquement — 16 zones)',        'Zone climatique CA'),
    ('in.energystar_climate_zone_2023',         'Zone climatique Energy Star 2023 (North/South/Marine)',         'Zone climatique'),
]

df_meteo = pd.DataFrame(meteo, columns=['Variable', 'Description', 'Catégorie'])
print('=== Variables Météo / Conditions Extérieures ===\n')
print(df_meteo.to_string(index=False))
print()
print('Variables météo ABSENTES dans metadata (disponibles uniquement dans les fichiers TMY) :')
print('  - Température extérieure horaire (°C)')
print('  - Humidité relative (%)')
print('  - Irradiation solaire (W/m²)')
print('  - Vitesse du vent (m/s)')
print('  → Ces données sont dans les fichiers .epw utilisés par EnergyPlus (non inclus dans ResStock parquet)')

# Distribution des zones climatiques
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ashrae = df_meta['in.ashrae_iecc_climate_zone_2004'].value_counts().sort_index()
colors_z = plt.cm.RdYlBu(np.linspace(0.1, 0.9, len(ashrae)))
bars = axes[0].bar(ashrae.index.astype(str), ashrae.values, color=colors_z, edgecolor='white')
for bar, val in zip(bars, ashrae.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                 f'{val/len(df_meta)*100:.1f}%', ha='center', fontsize=8, fontweight='bold')
axes[0].set_title('Zones ASHRAE IECC 2004\n(1=tropical → 8=subarctique)', fontweight='bold')
axes[0].set_xlabel('Zone climatique')
axes[0].set_ylabel('Nb bâtiments')

ba_zone = df_meta['in.building_america_climate_zone'].value_counts()
colors_ba = plt.cm.Set3(np.linspace(0, 1, len(ba_zone)))
axes[1].barh(ba_zone.index, ba_zone.values, color=colors_ba, edgecolor='white')
for bar, val in zip(axes[1].patches, ba_zone.values):
    axes[1].text(bar.get_width()+500, bar.get_y()+bar.get_height()/2,
                 f'{val/len(df_meta)*100:.1f}%', va='center', fontsize=8)
axes[1].set_title('Zones Building America (DOE)\nZones bioclimatiques', fontweight='bold')
axes[1].set_xlabel('Nb bâtiments')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Proxys Météo — Zones Climatiques ResStock', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'meteo_zones_climatiques.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Enveloppe Thermique
Variables décrivant l'isolation et la performance thermique du bâti.

In [ ]:
enveloppe = [
    # Isolation
    ('in.insulation_wall',            'Isolation murs extérieurs (R-value, ft²·°F·h/BTU)',      'Isolation'),
    ('in.insulation_ceiling',         'Isolation plafond/combles (R-value)',                     'Isolation'),
    ('in.insulation_roof',            'Isolation toiture (R-value)',                             'Isolation'),
    ('in.insulation_floor',           'Isolation plancher bas (R-value)',                        'Isolation'),
    ('in.insulation_foundation_wall', 'Isolation murs de fondation (R-value)',                   'Isolation'),
    ('in.insulation_rim_joist',       'Isolation solive de rive (R-value)',                      'Isolation'),
    ('in.insulation_slab',            'Isolation dalle béton (R-value)',                         'Isolation'),
    # Étanchéité
    ('in.air_leakage_to_outside_ach50','Infiltration d\'air (ACH50 — Volume/h à 50 Pa)',        'Étanchéité à l\'air'),
    ('in.infiltration',               'Niveau d\'infiltration (catégorie)',                      'Étanchéité à l\'air'),
    # Vitrage et ouvertures
    ('in.windows',                    'Type de vitrage (simple, double, triple + gaz)',          'Vitrage'),
    ('in.window_areas',               'Surface vitrée par orientation (ft²)',                   'Vitrage'),
    ('in.door_area',                  'Surface des portes (ft²)',                               'Ouvertures'),
    ('in.doors',                      'Type de porte (bois, fibre de verre...)',                 'Ouvertures'),
    # Protection solaire
    ('in.interior_shading',           'Protections solaires intérieures (stores, rideaux)',      'Protection solaire'),
    ('in.overhangs',                  'Avancées de toit (profondeur en ft)',                    'Protection solaire'),
    ('in.eaves',                      'Avant-toits (profondeur en ft)',                         'Protection solaire'),
    ('in.radiant_barrier',            'Barrière rayonnante en combles (Yes/No)',                 'Protection solaire'),
    # Géométrie
    ('in.geometry_floor_area',        'Surface habitable totale (m² après conversion)',          'Géométrie'),
    ('in.geometry_stories',           'Nombre d\'étages',                                       'Géométrie'),
    ('in.geometry_attic_type',        'Type de combles (conditioned, vented, unvented…)',        'Géométrie'),
    ('in.geometry_foundation_type',   'Type de fondation (slab, basement, crawlspace…)',         'Géométrie'),
    ('in.geometry_wall_type',         'Type de structure murale (wood frame, masonry…)',         'Matériaux'),
    ('in.geometry_wall_exterior_finish','Finition extérieure des murs (brique, bardage…)',       'Matériaux'),
    ('in.roof_material',              'Matériau de toiture (tuile, asphalte, métal…)',           'Matériaux'),
    # Thermique sol
    ('in.ground_thermal_conductivity','Conductivité thermique du sol (W/m·K)',                   'Thermique sol'),
    # Orientation
    ('in.orientation',                'Orientation du bâtiment (façade principale)',             'Orientation'),
    # Mitoyenneté (voir section 10)
    ('in.neighbors',                  'Configuration des voisins adjacents (mitoyenneté)',       'Mitoyenneté'),
]

df_env = pd.DataFrame(enveloppe, columns=['Variable', 'Description', 'Sous-catégorie'])
print('=== Enveloppe Thermique ===\n')
for cat, grp in df_env.groupby('Sous-catégorie'):
    print(f'\n  [{cat}]')
    for _, row in grp.iterrows():
        print(f'    {row["Variable"]:<45} {row["Description"]}')

# metadata_features.parquet a déjà converti "R-11" → 11.0 (fait dans transformations_numeriques.ipynb)
df_feat = pd.read_parquet(DATA_PROCESSED / 'metadata_features.parquet')

insul_cols = ['in.insulation_wall', 'in.insulation_ceiling', 'in.insulation_roof',
              'in.insulation_floor', 'in.insulation_slab']
palette    = ['#3b82f6', '#22c55e', '#f59e0b', '#ef4444', '#8b5cf6']

data_insul   = []
labels_insul = []
colors_used  = []

for col, color in zip(insul_cols, palette):
    if col in df_feat.columns:
        vals = pd.to_numeric(df_feat[col], errors='coerce').dropna()
        if len(vals) > 1000:
            data_insul.append(vals.values)
            labels_insul.append(col.replace('in.insulation_', ''))
            colors_used.append(color)

fig, ax = plt.subplots(figsize=(12, 5))
bplot = ax.boxplot(data_insul, labels=labels_insul, patch_artist=True,
                   medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bplot['boxes'], colors_used):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Distribution des R-values d\'isolation par élément constructif', fontweight='bold', fontsize=12)
ax.set_ylabel('R-value (ft²·°F·h/BTU)\n[plus élevé = mieux isolé]')
ax.set_xlabel('Élément constructif')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / 'enveloppe_r_values.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Systèmes Énergétiques (HVAC + Ventilation + Eau Chaude)

In [ ]:
systemes = [
    # --- VENTILATION ---
    ('in.mechanical_ventilation',    'Type de VMC : None / ERV / HRV / exhaust only…',         'Ventilation'),
    ('in.natural_ventilation',       'Ventilation naturelle (ouverture fenêtres) — proxy comportemental', 'Ventilation'),
    ('in.bathroom_spot_vent_hour',   'Heure d\'activation ventilation salle de bain (0–23h)',   'Ventilation'),
    ('in.range_spot_vent_hour',      'Heure d\'activation hotte cuisine (0–23h)',               'Ventilation'),
    ('in.dehumidifier',              'Présence d\'un déshumidificateur',                        'Ventilation'),
    # --- CHAUFFAGE ---
    ('in.hvac_heating_type',         'Type de chauffage (Furnace, Boiler, ASHP, Baseboard…)',  'Chauffage'),
    ('in.hvac_heating_type_and_fuel','Type de chauffage + combustible combiné',                 'Chauffage'),
    ('in.hvac_heating_efficiency',   'Rendement chauffage (AFUE % ou HSPF selon type)',         'Chauffage'),
    ('in.hvac_heating_autosizing_factor', 'Facteur de dimensionnement du chauffage',            'Chauffage'),
    ('in.hvac_has_zonal_electric_heating', 'Présence chauffage électrique zonal (convecteur)',  'Chauffage'),
    ('in.hvac_secondary_heating_type','Type de chauffage secondaire (appoint)',                 'Chauffage'),
    ('in.hvac_secondary_heating_fuel','Combustible du chauffage secondaire',                    'Chauffage'),
    ('in.hvac_secondary_heating_efficiency', 'Rendement chauffage secondaire',                  'Chauffage'),
    ('in.heating_fuel',              'Combustible principal du chauffage (Electricity, Gas…)', 'Chauffage'),
    ('in.heating_unavailable_days',  'Nb jours sans chauffage (0 = toujours disponible)',      'Chauffage'),
    ('in.heating_unavailable_period','Période d\'indisponibilité du chauffage',                'Chauffage'),
    # --- CLIMATISATION ---
    ('in.hvac_cooling_type',         'Type de climatisation (Central AC, Room AC, None…)',     'Climatisation'),
    ('in.hvac_cooling_efficiency',   'Rendement climatisation (SEER ou EER)',                   'Climatisation'),
    ('in.hvac_cooling_autosizing_factor', 'Facteur de dimensionnement clim',                    'Climatisation'),
    ('in.hvac_cooling_partial_space_conditioning', '% de l\'espace climatisé',                 'Climatisation'),
    ('in.cooling_unavailable_days',  'Nb jours sans climatisation',                            'Climatisation'),
    ('in.cooling_unavailable_period','Période d\'indisponibilité de la clim',                  'Climatisation'),
    # --- GAINES ---
    ('in.hvac_has_ducts',            'Présence de gaines de distribution (Yes/No)',             'Distribution air'),
    ('in.duct_leakage_and_insulation','Fuites gaines (%) + isolation (R-value)',                'Distribution air'),
    ('in.duct_location',             'Localisation gaines (combles, sous-sol, espace de vie)', 'Distribution air'),
    ('in.hvac_has_shared_system',    'Système HVAC partagé entre logements',                    'Distribution air'),
    ('in.hvac_shared_efficiencies',  'Rendements du système HVAC partagé',                     'Distribution air'),
    # --- PARAMÈTRES TECHNIQUES ---
    ('in.hvac_system_is_faulted',    'Système HVAC en défaut (anomalie de fonctionnement)',     'Paramètres techniques'),
    ('in.hvac_system_is_scaled',     'Système dimensionné selon des règles d\'échelle',         'Paramètres techniques'),
    ('in.hvac_system_single_speed_ac_airflow',  'Débit d\'air clim monophasé',                 'Paramètres techniques'),
    ('in.hvac_system_single_speed_ac_charge',   'Charge frigorigène clim monophasée',           'Paramètres techniques'),
    ('in.hvac_system_single_speed_ashp_airflow','Débit d\'air PAC monophasée',                 'Paramètres techniques'),
    ('in.hvac_system_single_speed_ashp_charge', 'Charge frigorigène PAC monophasée',            'Paramètres techniques'),
    # --- EAU CHAUDE SANITAIRE ---
    ('in.water_heater_fuel',         'Combustible chauffe-eau (Electricity, Gas, Heat Pump…)', 'Eau Chaude (DHW)'),
    ('in.water_heater_efficiency',   'Rendement chauffe-eau (EF = Energy Factor)',              'Eau Chaude (DHW)'),
    ('in.water_heater_in_unit',      'Chauffe-eau dans le logement (Yes/No)',                  'Eau Chaude (DHW)'),
    ('in.water_heater_location',     'Localisation chauffe-eau (conditioned / unconditioned)', 'Eau Chaude (DHW)'),
    ('in.hot_water_distribution',    'Type de distribution ECS (standard, réservoir, recirculation)', 'Eau Chaude (DHW)'),
    ('in.hot_water_fixtures',        'Niveau d\'utilisation robinetterie (60%–150% Usage)',    'Eau Chaude (DHW)'),
]

df_sys = pd.DataFrame(systemes, columns=['Variable', 'Description', 'Sous-catégorie'])
print('=== Systèmes Énergétiques ===\n')
for cat, grp in df_sys.groupby('Sous-catégorie'):
    print(f'  [{cat}] — {len(grp)} variables')
    for _, row in grp.iterrows():
        print(f'    {row["Variable"]}')

# Visualisation types HVAC
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

heat_types = df_meta['in.hvac_heating_type'].value_counts().head(10)
axes[0].barh(heat_types.index, heat_types.values, color='#ef4444', alpha=0.8, edgecolor='white')
axes[0].invert_yaxis()
for bar, val in zip(axes[0].patches, heat_types.values):
    axes[0].text(bar.get_width()+500, bar.get_y()+bar.get_height()/2,
                 f'{val/len(df_meta)*100:.1f}%', va='center', fontsize=8)
axes[0].set_title('Types de chauffage (top 10)', fontweight='bold')
axes[0].set_xlabel('Nb bâtiments')

cool_types = df_meta['in.hvac_cooling_type'].value_counts().head(8)
axes[1].barh(cool_types.index, cool_types.values, color='#3b82f6', alpha=0.8, edgecolor='white')
axes[1].invert_yaxis()
for bar, val in zip(axes[1].patches, cool_types.values):
    axes[1].text(bar.get_width()+500, bar.get_y()+bar.get_height()/2,
                 f'{val/len(df_meta)*100:.1f}%', va='center', fontsize=8)
axes[1].set_title('Types de climatisation', fontweight='bold')
axes[1].set_xlabel('Nb bâtiments')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Systèmes HVAC — Chauffage et Climatisation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'hvac_systemes_classification.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Classification des Usages Électriques (`out.electricity.*`)
Attribution de chaque usage à une catégorie énergétique métier.

In [ ]:
TOTAL_COL = 'out.electricity.total.energy_consumption..kwh'

usages_classification = [
    # Chauffage
    ('out.electricity.heating.energy_consumption..kwh',         'Chauffage',         'Chauffage électrique principal (résistif ou PAC)'),
    ('out.electricity.heating_fans_pumps.energy_consumption..kwh','Chauffage',       'Ventilateurs et pompes du circuit de chauffage'),
    ('out.electricity.heating_hp_bkup.energy_consumption..kwh', 'Chauffage',         'Appoint électrique de la pompe à chaleur'),
    ('out.electricity.heating_hp_bkup_fa.energy_consumption..kwh','Chauffage',       'Appoint PAC — fan assisted'),
    # Climatisation
    ('out.electricity.cooling.energy_consumption..kwh',         'Climatisation',     'Climatisation principale (compresseur)'),
    ('out.electricity.cooling_fans_pumps.energy_consumption..kwh','Climatisation',   'Ventilateurs et pompes du circuit de climatisation'),
    # Eau Chaude Sanitaire (DHW)
    ('out.electricity.hot_water.energy_consumption..kwh',       'DHW',               'Chauffe-eau électrique ou PAC eau chaude'),
    ('out.electricity.hot_water_solar_th.energy_consumption..kwh','DHW',             'Appoint thermique solaire eau chaude'),
    # Blanc — électroménager
    ('out.electricity.clothes_dryer.energy_consumption..kwh',   'Blanc',             'Sèche-linge'),
    ('out.electricity.clothes_washer.energy_consumption..kwh',  'Blanc',             'Lave-linge'),
    ('out.electricity.dishwasher.energy_consumption..kwh',      'Blanc',             'Lave-vaisselle'),
    ('out.electricity.refrigerator.energy_consumption..kwh',    'Blanc',             'Réfrigérateur'),
    ('out.electricity.freezer.energy_consumption..kwh',         'Blanc',             'Congélateur (extra)'),
    ('out.electricity.range_oven.energy_consumption..kwh',      'Blanc',             'Cuisinière / Four électrique'),
    # Brun — électronique
    ('out.electricity.television.energy_consumption..kwh',      'Brun',              'Téléviseurs'),
    ('out.electricity.plug_loads.energy_consumption..kwh',      'Brun',              'Charges prises (ordinateurs, chargeurs, multimédia…)'),
    # Éclairage
    ('out.electricity.lighting_interior.energy_consumption..kwh','Éclairage',        'Éclairage intérieur'),
    ('out.electricity.lighting_exterior.energy_consumption..kwh','Éclairage',        'Éclairage extérieur'),
    ('out.electricity.lighting_garage.energy_consumption..kwh', 'Éclairage',         'Éclairage garage'),
    ('out.electricity.ceiling_fan.energy_consumption..kwh',     'Éclairage',         'Ventilateurs de plafond'),
    # Ventilation
    ('out.electricity.mech_vent.energy_consumption..kwh',       'Ventilation',       'Ventilation mécanique (VMC, ERV, HRV)'),
    # Divers / Usages spéciaux
    ('out.electricity.well_pump.energy_consumption..kwh',       'Divers',            'Pompe de puits'),
    ('out.electricity.pool_heater.energy_consumption..kwh',     'Divers',            'Chauffage piscine'),
    ('out.electricity.pool_pump.energy_consumption..kwh',       'Divers',            'Pompe piscine'),
    ('out.electricity.permanent_spa_heat.energy_consumption..kwh','Divers',          'Chauffage spa/jacuzzi'),
    ('out.electricity.permanent_spa_pump.energy_consumption..kwh','Divers',          'Pompe spa/jacuzzi'),
    # VE (exclu de l'analyse principale — voir section 8)
    ('out.electricity.ev_charging.energy_consumption..kwh',     'VE — EXCLU',        'Recharge véhicule électrique'),
    # Production
    ('out.electricity.pv.energy_consumption..kwh',              'Production PV',     'Production photovoltaïque (valeur négative = injection)'),
    # Totaux
    ('out.electricity.net.energy_consumption..kwh',             'Total net',         'Consommation nette (après déduction PV)'),
    ('out.electricity.total.energy_consumption..kwh',           'Total brut',        'Consommation électrique totale'),
]

df_usages = pd.DataFrame(usages_classification,
                          columns=['Variable', 'Catégorie', 'Description'])

print('=== Classification des usages électriques ===\n')
for cat, grp in df_usages.groupby('Catégorie'):
    conso_cols = [c for c in grp['Variable'] if c in df_raw.columns]
    if conso_cols:
        mean_conso = df_raw[conso_cols].sum(axis=1).mean()
        pct = mean_conso / df_raw[TOTAL_COL].mean() * 100 if cat not in ['VE — EXCLU','Total net','Total brut','Production PV'] else 0
        print(f'  [{cat}] {mean_conso:8.0f} kWh/an   {pct:5.1f}% du total')
        for _, row in grp.iterrows():
            print(f'    {row["Variable"]:<60} → {row["Description"]}')
    else:
        print(f'  [{cat}]')

print(f'\n  Consommation totale moyenne : {df_raw[TOTAL_COL].mean():,.0f} kWh/an')

---
## 7. Part des Usages Blanc et Brun
Calcul et visualisation de la décomposition des usages spécifiques.

In [ ]:
BLANC_COLS = [
    'out.electricity.clothes_dryer.energy_consumption..kwh',
    'out.electricity.clothes_washer.energy_consumption..kwh',
    'out.electricity.dishwasher.energy_consumption..kwh',
    'out.electricity.refrigerator.energy_consumption..kwh',
    'out.electricity.freezer.energy_consumption..kwh',
    'out.electricity.range_oven.energy_consumption..kwh',
]
BRUN_COLS = [
    'out.electricity.television.energy_consumption..kwh',
    'out.electricity.plug_loads.energy_consumption..kwh',
]

blanc_cols_ok = [c for c in BLANC_COLS if c in df_raw.columns]
brun_cols_ok  = [c for c in BRUN_COLS  if c in df_raw.columns]

conso_blanc = df_raw[blanc_cols_ok].sum(axis=1)
conso_brun  = df_raw[brun_cols_ok].sum(axis=1)
conso_total = df_raw[TOTAL_COL]

pct_blanc = (conso_blanc / conso_total * 100).median()
pct_brun  = (conso_brun  / conso_total * 100).median()

print('=== Part des Usages Blanc et Brun ===\n')
print(f'  Conso BLANC moyenne : {conso_blanc.mean():8.0f} kWh/an  ({conso_blanc.mean()/conso_total.mean()*100:.1f}% du total)')
for col in blanc_cols_ok:
    label = col.replace('out.electricity.','').replace('.energy_consumption..kwh','')
    print(f'    {label:<35} {df_raw[col].mean():7.0f} kWh/an')

print(f'\n  Conso BRUN moyenne  : {conso_brun.mean():8.0f} kWh/an  ({conso_brun.mean()/conso_total.mean()*100:.1f}% du total)')
for col in brun_cols_ok:
    label = col.replace('out.electricity.','').replace('.energy_consumption..kwh','')
    print(f'    {label:<35} {df_raw[col].mean():7.0f} kWh/an')

print(f'\n  Médiane % Blanc : {pct_blanc:.1f}%')
print(f'  Médiane % Brun  : {pct_brun:.1f}%')
print(f'  Note : plug_loads inclut TV + ordinateurs + chargeurs, donc Brun sous-estimé')

# Visualisation décomposition complète
categories_plot = {
    'Chauffage':      ['out.electricity.heating.energy_consumption..kwh',
                       'out.electricity.heating_fans_pumps.energy_consumption..kwh',
                       'out.electricity.heating_hp_bkup.energy_consumption..kwh'],
    'Climatisation':  ['out.electricity.cooling.energy_consumption..kwh',
                       'out.electricity.cooling_fans_pumps.energy_consumption..kwh'],
    'DHW':            ['out.electricity.hot_water.energy_consumption..kwh'],
    'Blanc':          blanc_cols_ok,
    'Brun':           brun_cols_ok,
    'Éclairage':      ['out.electricity.lighting_interior.energy_consumption..kwh',
                       'out.electricity.lighting_exterior.energy_consumption..kwh',
                       'out.electricity.ceiling_fan.energy_consumption..kwh'],
    'Ventilation':    ['out.electricity.mech_vent.energy_consumption..kwh'],
    'VE':             ['out.electricity.ev_charging.energy_consumption..kwh'],
    'Divers':         ['out.electricity.well_pump.energy_consumption..kwh',
                       'out.electricity.pool_pump.energy_consumption..kwh',
                       'out.electricity.pool_heater.energy_consumption..kwh'],
}

moyennes = {}
for cat, cols in categories_plot.items():
    cols_ok = [c for c in cols if c in df_raw.columns]
    if cols_ok:
        moyennes[cat] = df_raw[cols_ok].sum(axis=1).mean()

colors_cat = {
    'Chauffage':'#ef4444', 'Climatisation':'#3b82f6', 'DHW':'#f97316',
    'Blanc':'#22c55e', 'Brun':'#8b5cf6', 'Éclairage':'#eab308',
    'Ventilation':'#06b6d4', 'VE':'#64748b', 'Divers':'#94a3b8'
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Camembert
labels_p = list(moyennes.keys())
values_p = list(moyennes.values())
colors_p = [colors_cat[l] for l in labels_p]
wedges, texts, autotexts = ax1.pie(
    values_p, labels=labels_p, colors=colors_p,
    autopct=lambda p: f'{p:.1f}%' if p > 2 else '',
    startangle=90, pctdistance=0.8
)
for at in autotexts: at.set_fontsize(9)
ax1.set_title('Décomposition conso électrique moyenne\n(549 971 bâtiments)', fontweight='bold', fontsize=11)

# Barres horizontales avec valeurs
sorted_items = sorted(moyennes.items(), key=lambda x: x[1], reverse=True)
names, vals = zip(*sorted_items)
bars = ax2.barh(names, vals, color=[colors_cat[n] for n in names], edgecolor='white', alpha=0.85)
for bar, val in zip(bars, vals):
    pct = val / conso_total.mean() * 100
    ax2.text(bar.get_width()+50, bar.get_y()+bar.get_height()/2,
             f'{val:,.0f} kWh  ({pct:.1f}%)', va='center', fontsize=9)
ax2.set_title('Consommation moyenne par catégorie (kWh/an)', fontweight='bold', fontsize=11)
ax2.set_xlabel('kWh/an')
ax2.invert_yaxis()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Classification des Usages Électriques — Blanc, Brun et autres', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'usages_blanc_brun_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Variables Véhicule Électrique — Exclusion de l'Analyse Principale
Les VE constituent un usage spécifique, non lié aux besoins thermiques du logement.  
Ils doivent être **isolés** pour ne pas biaiser les analyses de flexibilité HVAC.

In [ ]:
ve_vars = [
    ('in.electric_vehicle_ownership',        'Ménage possède un VE (Yes/No)'),
    ('in.electric_vehicle_charger',          'Type de chargeur (Level 1 / Level 2 / None)'),
    ('in.electric_vehicle_charge_at_home',   'Recharge à domicile (Yes/No)'),
    ('in.electric_vehicle_battery',          'Capacité batterie VE (kWh)'),
    ('in.electric_vehicle_miles_traveled',   'Kilométrage annuel VE (miles)'),
    ('in.electric_vehicle_outlet_access',    'Accès à une prise dédiée VE (Yes/No)'),
    ('out.electricity.ev_charging.energy_consumption..kwh', 'Consommation annuelle recharge VE (kWh)'),
]

df_ve = pd.DataFrame(ve_vars, columns=['Variable', 'Description'])
print('=== Variables VE à exclure de l\'analyse thermique principale ===\n')
print(df_ve.to_string(index=False))

pct_ev_owner = (df_meta['in.electric_vehicle_ownership'] == 'Yes').mean() * 100
pct_ev_home  = (df_meta['in.electric_vehicle_charge_at_home'] == 'Yes').mean() * 100
ev_conso_mean = df_raw['out.electricity.ev_charging.energy_consumption..kwh'].mean()
ev_conso_cond = df_raw.loc[df_raw['out.electricity.ev_charging.energy_consumption..kwh'] > 0,
                            'out.electricity.ev_charging.energy_consumption..kwh'].mean()

print(f'\nStats VE dans le dataset :')
print(f'  Propriétaires de VE      : {pct_ev_owner:.1f}% des ménages')
print(f'  Rechargent à domicile    : {pct_ev_home:.1f}% des ménages')
print(f'  Conso recharge moyenne (tous) : {ev_conso_mean:.0f} kWh/an')
print(f'  Conso recharge (possesseurs VE seulement) : {ev_conso_cond:.0f} kWh/an')
print(f'\n  → Faible pénétration (1.1%) mais conso VE élevée (~{ev_conso_cond:.0f} kWh/an)')
print(f'  → Flexibilité VE = usage pilotable majeur pour les ménages équipés')
print(f'  → À traiter dans une analyse dédiée séparée (hors HVAC)')

---
## 9. Variables Comportementales
Variables décrivant les réglages, habitudes et comportements des occupants vis-à-vis des équipements.

In [ ]:
comportement = [
    # Thermostat
    ('in.heating_setpoint',                 'Consigne de chauffage (°C après conversion)',            'Thermostat'),
    ('in.cooling_setpoint',                 'Consigne de climatisation (°C après conversion)',        'Thermostat'),
    ('in.heating_setpoint_has_offset',      'Thermostat de chauffage programmable (Yes/No)',          'Thermostat'),
    ('in.cooling_setpoint_has_offset',      'Thermostat de clim programmable (Yes/No)',               'Thermostat'),
    ('in.heating_setpoint_offset_magnitude','Amplitude du décalage thermostat chauffage (°C)',        'Thermostat'),
    ('in.cooling_setpoint_offset_magnitude','Amplitude du décalage thermostat clim (°C)',             'Thermostat'),
    ('in.heating_setpoint_offset_period',   'Plage horaire du décalage thermostat chauffage',         'Thermostat'),
    ('in.cooling_setpoint_offset_period',   'Plage horaire du décalage thermostat clim',              'Thermostat'),
    # Niveau d'utilisation global
    ('in.usage_level',                      'Niveau global (Low/Med/High) — multiplie tous usages',   'Niveau usage'),
    # Appareils
    ('in.clothes_dryer_usage_level',        'Intensité usage sèche-linge (80/100/120%)',              'Appareils'),
    ('in.clothes_washer_usage_level',       'Intensité usage lave-linge (80/100/120%)',               'Appareils'),
    ('in.dishwasher_usage_level',           'Intensité usage lave-vaisselle (80/100/120%)',           'Appareils'),
    ('in.cooking_range_usage_level',        'Intensité usage cuisinière (80/100/120%)',               'Appareils'),
    ('in.refrigerator_usage_level',         'Intensité usage réfrigérateur (95/100/105%)',            'Appareils'),
    ('in.plug_loads',                       'Niveau de charge des prises (%)',                        'Appareils'),
    ('in.plug_load_diversity',              'Diversité temporelle des charges (50/100/200%)',          'Appareils'),
    # Eau chaude
    ('in.hot_water_fixtures',               'Usage robinetterie eau chaude (60–150% Usage)',          'Eau chaude'),
    # Éclairage
    ('in.lighting_interior_use',            'Niveau usage éclairage intérieur (100% Usage)',          'Éclairage'),
    ('in.lighting_other_use',               'Niveau usage éclairage autre (100% Usage)',              'Éclairage'),
    ('in.holiday_lighting',                 'Présence éclairage de fête (Thanksgiving/Christmas…)',   'Éclairage'),
    # Ventilation comportementale
    ('in.natural_ventilation',              'Comportement ouverture fenêtres (proxy)',                'Ventilation'),
    ('in.bathroom_spot_vent_hour',          'Heure déclenchement ventilation salle de bain',          'Ventilation'),
    ('in.range_spot_vent_hour',             'Heure déclenchement hotte cuisine',                     'Ventilation'),
    # Indisponibilité systèmes
    ('in.heating_unavailable_days',         'Jours thermostat chauffage coupé (comportement)',        'Disponibilité'),
    ('in.heating_unavailable_period',       'Période sans chauffage (printemps/automne)',             'Disponibilité'),
    ('in.cooling_unavailable_days',         'Jours thermostat clim coupé (comportement)',             'Disponibilité'),
    ('in.cooling_unavailable_period',       'Période sans climatisation',                            'Disponibilité'),
]

df_comport = pd.DataFrame(comportement, columns=['Variable', 'Description', 'Sous-catégorie'])
print('=== Variables Comportementales ===\n')
for cat, grp in df_comport.groupby('Sous-catégorie'):
    print(f'  [{cat}]')
    for _, row in grp.iterrows():
        print(f'    {row["Variable"]:<50} {row["Description"]}')

# Visualisation setpoints (consignes thermostat)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label, color in [
    (axes[0], 'in.heating_setpoint', 'Consigne chauffage (°C)', '#ef4444'),
    (axes[1], 'in.cooling_setpoint', 'Consigne climatisation (°C)', '#3b82f6'),
]:
    vals = pd.to_numeric(df_meta[col], errors='coerce').dropna()
    ax.hist(vals, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(vals.median(), color='black', linestyle='--',
               label=f'Médiane = {vals.median():.1f}°C')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('°C')
    ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Consignes Thermostat — Variables Comportementales Clés', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'comportement_setpoints.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Analyse de `in.neighbors` — Indicateur de Mitoyenneté
Cette variable décrit la **configuration des bâtiments voisins adjacents**, pas les conditions météo.

In [ ]:
print('=== Analyse de in.neighbors ===\n')
print('Valeurs présentes dans le dataset :')
neigh_vc = df_meta['in.neighbors'].value_counts()
for val, cnt in neigh_vc.items():
    print(f'  {str(val):<30} : {cnt:6d}  ({cnt/len(df_meta)*100:.1f}%)')

print()
print('Interprétation :')
print('  "Left/Right at 15ft" (67.7%) → Maison individuelle avec voisins à 15 pieds de chaque côté')
print('  2, 4, 7, 12, 27             → Immeuble collectif — nombre d\'unités par bâtiment')
print('  None (1.3%)                 → Bâtiment isolé, pas de voisins adjacents')

print()
print('Est-ce un proxy météo ?')
print('  NON — in.neighbors n\'est PAS une variable météo.')
print('  C\'est une variable d\'ENVELOPPE THERMIQUE (mitoyenneté) :')
print('  → Un bâtiment mitoyen a moins de surface exposée au froid/soleil')
print('  → Impact direct : réduction des déperditions thermiques par les murs mitoyens')
print('  → Un appartement au cœur d\'un immeuble (12+ voisins) perd peu de chaleur')
print('    versus un pavillon isolé qui perd de la chaleur par les 4 côtés')

# Corrélation avec la consommation de chauffage
df_combined = df_meta[['in.neighbors']].copy()
df_combined['heating'] = df_raw['out.electricity.heating.energy_consumption..kwh'].values
df_combined['total']   = df_raw['out.electricity.total.energy_consumption..kwh'].values

mean_heat = df_combined.groupby('in.neighbors')['heating'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors_n = ['#3b82f6' if str(v) == 'None' else
            '#22c55e' if str(v) == 'Left/Right at 15ft' else
            '#f97316' for v in mean_heat.index]
bars = ax.barh([str(v) for v in mean_heat.index], mean_heat.values,
               color=colors_n, edgecolor='white', alpha=0.85)
for bar, val in zip(bars, mean_heat.values):
    ax.text(bar.get_width()+10, bar.get_y()+bar.get_height()/2,
            f'{val:,.0f} kWh', va='center', fontsize=9)
ax.set_title('Consommation chauffage moyenne selon la configuration des voisins\n'
             '(orange = immeuble collectif, vert = maison mitoyenne, bleu = isolé)',
             fontweight='bold')
ax.set_xlabel('Consommation chauffage électrique (kWh/an)')
ax.set_ylabel('in.neighbors')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / 'neighbors_chauffage.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nConclusion : in.neighbors est une variable d\'ENVELOPPE THERMIQUE')
print('  → À classer dans le groupe "Géométrie / Mitoyenneté"')
print('  → Corrélation NÉGATIVE avec le chauffage : plus de voisins = moins de chauffage')

---
## 11. Tableau Récapitulatif — Toutes Variables Classées

In [ ]:
toutes_variables = [
    # === SOCIO-ÉCONOMIQUE ===
    ('in.income',                    'Socio-économique',  'Conserver',  'Revenu du ménage — corrélé à la qualité du bâtiment et aux comportements'),
    ('in.representative_income',     'Socio-économique',  'Conserver',  'Version pondérée du revenu'),
    ('in.federal_poverty_level',     'Socio-économique',  'Conserver',  'Seuil de pauvreté — indicateur précarité énergétique'),
    ('in.area_median_income',        'Socio-économique',  'Conserver',  'Revenu médian zone — contexte économique local'),
    ('in.state_metro_median_income', 'Socio-économique',  'Supprimer',  'Redondant avec area_median_income'),
    ('in.tenure',                    'Socio-économique',  'Conserver',  'Propriétaire vs locataire — influence les rénovations'),
    ('in.household_has_tribal_persons','Socio-économique','Supprimer',  'Faible impact énergétique direct'),
    ('in.aiannh_area',               'Socio-économique',  'Supprimer',  'Zone tribale — peu informatif pour la flexibilité'),
    ('in.units_represented',         'Socio-économique',  'Supprimer',  'Pondération statistique, pas une feature énergétique'),
    # === OCCUPANTS ===
    ('in.occupants',                 'Occupants',         'Conserver',  'Apports internes — forte influence sur le chauffage hivernal'),
    ('in.bedrooms',                  'Occupants',         'Conserver',  'Proxy taille foyer'),
    ('in.vacancy_status',            'Occupants',         'Conserver',  'Logement vacant = 0 apport interne, comportement absent'),
    ('in.usage_level',               'Occupants',         'Conserver',  'Multiplie tous les usages — variable comportementale globale'),
    # === MÉTÉO (proxys) ===
    ('in.ashrae_iecc_climate_zone_2004','Météo',          'Conserver',  'Variable clé — principale classification climatique ResStock'),
    ('in.building_america_climate_zone','Météo',          'Conserver',  'Complément — bioclimatique'),
    ('in.ashrae_iecc_climate_zone_2004_sub_cz_split','Météo','Conserver','Sous-zone A/B/C — humide/sec/marin'),
    ('in.weather_file_city',         'Météo',             'Supprimer',  'Redondant avec lat/lon — trop de modalités'),
    ('in.weather_file_latitude',     'Météo',             'Conserver',  'Latitude — proxy ensoleillement et altitude thermique'),
    ('in.weather_file_longitude',    'Météo',             'Conserver',  'Longitude — contexte géographique'),
    ('in.cec_climate_zone',          'Météo',             'Supprimer',  'Californie uniquement — trop local'),
    ('in.energystar_climate_zone_2023','Météo',           'Supprimer',  'Redondant avec ASHRAE'),
    # === ENVELOPPE THERMIQUE ===
    ('in.insulation_wall',           'Enveloppe',         'Conserver',  'R-value murs — fort impact sur les déperditions'),
    ('in.insulation_ceiling',        'Enveloppe',         'Conserver',  'R-value plafond/combles — 57% NaN (type de bâtiment)'),
    ('in.insulation_roof',           'Enveloppe',         'Conserver',  'R-value toiture'),
    ('in.insulation_floor',          'Enveloppe',         'Conserver',  'R-value plancher — 39% NaN'),
    ('in.insulation_foundation_wall','Enveloppe',         'Conserver',  'R-value fondations — 48% NaN'),
    ('in.insulation_rim_joist',      'Enveloppe',         'Conserver',  'R-value solive — 48% NaN'),
    ('in.insulation_slab',           'Enveloppe',         'Conserver',  'R-value dalle — 61% NaN'),
    ('in.air_leakage_to_outside_ach50','Enveloppe',       'Conserver',  'Étanchéité à l\'air — variable continue très informative'),
    ('in.infiltration',              'Enveloppe',         'Supprimer',  'Redondant avec air_leakage_to_outside_ach50'),
    ('in.windows',                   'Enveloppe',         'Conserver',  'Type vitrage — U-value impliqué'),
    ('in.window_areas',              'Enveloppe',         'Conserver',  'Surface vitrée par orientation'),
    ('in.door_area',                 'Enveloppe',         'Conserver',  'Surface portes'),
    ('in.doors',                     'Enveloppe',         'Supprimer',  'Type porte — faible impact vs fenêtres'),
    ('in.interior_shading',          'Enveloppe',         'Conserver',  'Protection solaire — réduit les gains solaires été'),
    ('in.overhangs',                 'Enveloppe',         'Supprimer',  'Avancées toit — peu discriminant'),
    ('in.eaves',                     'Enveloppe',         'Supprimer',  'Avant-toits — corrélé à overhangs'),
    ('in.radiant_barrier',           'Enveloppe',         'Conserver',  'Barrière rayonnante combles — impact été zones chaudes'),
    ('in.geometry_floor_area',       'Enveloppe',         'Conserver',  'Surface habitable — driver principal de la conso totale'),
    ('in.geometry_stories',          'Enveloppe',         'Conserver',  'Étages — influence forme bâtiment et ratio surface/volume'),
    ('in.geometry_attic_type',       'Enveloppe',         'Conserver',  'Combles conditionné ou non — impact thermique'),
    ('in.geometry_foundation_type',  'Enveloppe',         'Conserver',  'Type fondation — déperditions plancher'),
    ('in.geometry_wall_type',        'Enveloppe',         'Conserver',  'Structure murale — inertie thermique'),
    ('in.geometry_wall_exterior_finish','Enveloppe',      'Supprimer',  'Finition extérieure — peu d\'impact thermique direct'),
    ('in.roof_material',             'Enveloppe',         'Conserver',  'Matériau toiture — albédo, réflectivité solaire'),
    ('in.ground_thermal_conductivity','Enveloppe',        'Supprimer',  'Conductivité sol — très peu discriminant'),
    ('in.orientation',               'Enveloppe',         'Conserver',  'Orientation — gains solaires passifs'),
    ('in.neighbors',                 'Enveloppe',         'Conserver',  'Mitoyenneté — réduit les déperditions latérales'),
    # === HVAC / CHAUFFAGE ===
    ('in.hvac_heating_type',         'HVAC',              'Conserver',  'Type chauffage — variable clé pour la flexibilité'),
    ('in.hvac_heating_type_and_fuel','HVAC',              'Supprimer',  'Redondant avec heating_type + heating_fuel'),
    ('in.hvac_heating_efficiency',   'HVAC',              'Conserver',  'Rendement chauffage — COP ou AFUE'),
    ('in.hvac_cooling_type',         'HVAC',              'Conserver',  'Type climatisation'),
    ('in.hvac_cooling_efficiency',   'HVAC',              'Conserver',  'SEER — rendement clim'),
    ('in.heating_fuel',              'HVAC',              'Conserver',  'Combustible — discrimine chauffage électrique vs gaz'),
    ('in.hvac_has_ducts',            'HVAC',              'Conserver',  'Gaines — distribution air forcé'),
    ('in.duct_leakage_and_insulation','HVAC',             'Conserver',  'Fuites gaines — pertes thermiques distribution'),
    ('in.duct_location',             'HVAC',              'Conserver',  'Localisation gaines — impact thermique si combles'),
    ('in.hvac_secondary_heating_type','HVAC',             'Conserver',  'Chauffage appoint — important pour flexibilité'),
    ('in.mechanical_ventilation',    'HVAC',              'Conserver',  'VMC — énergie ventilation et qualité air'),
    ('in.hvac_system_is_faulted',    'HVAC',              'Supprimer',  'Quasi-constant (0.0% Yes) — aucune variance'),
    ('in.hvac_system_is_scaled',     'HVAC',              'Supprimer',  'Quasi-constant — aucune variance'),
    # === EAU CHAUDE ===
    ('in.water_heater_fuel',         'DHW',               'Conserver',  'Combustible chauffe-eau — électrique pilotable'),
    ('in.water_heater_efficiency',   'DHW',               'Conserver',  'Rendement EF'),
    ('in.water_heater_in_unit',      'DHW',               'Conserver',  '85.7% Yes — sinon chauffe-eau partagé'),
    ('in.water_heater_location',     'DHW',               'Conserver',  'Localisation — impact si espace non conditionné'),
    ('in.hot_water_distribution',    'DHW',               'Conserver',  'Type distribution ECS'),
    # === COMPORTEMENTAL ===
    ('in.heating_setpoint',          'Comportemental',    'Conserver',  'Consigne chauffage — variable de flexibilité directe'),
    ('in.cooling_setpoint',          'Comportemental',    'Conserver',  'Consigne clim — variable de flexibilité directe'),
    ('in.heating_setpoint_has_offset','Comportemental',   'Conserver',  'Thermostat programmable — 43.8% Yes'),
    ('in.cooling_setpoint_has_offset','Comportemental',   'Conserver',  'Thermostat clim programmable — 35.8% Yes'),
    ('in.heating_setpoint_offset_magnitude','Comportemental','Conserver','Amplitude décalage thermostat — quantifie la flexibilité'),
    ('in.usage_level',               'Comportemental',    'Conserver',  'Multiplie tous les usages'),
    ('in.natural_ventilation',       'Comportemental',    'Conserver',  'Ouverture fenêtres — comportement estival'),
    # === VE — EXCLU ===
    ('in.electric_vehicle_ownership','VE — Exclu',        'Traiter séparément', 'Faible pénétration (1.1%) — analyse dédiée'),
    ('in.electric_vehicle_charge_at_home','VE — Exclu',   'Traiter séparément', 'Usage pilotable mais non thermique'),
    ('in.electric_vehicle_battery',  'VE — Exclu',        'Traiter séparément', 'Capacité stockage'),
    ('in.electric_vehicle_miles_traveled','VE — Exclu',   'Traiter séparément', 'Kilométrage annuel'),
    # === ÉNERGIE RENOUVELABLE ===
    ('in.has_pv',                    'Production',        'Conserver',  'Présence PV — 1.0% des bâtiments'),
    ('in.pv_system_size',            'Production',        'Conserver',  'Taille système PV (kW)'),
    ('in.pv_orientation',            'Production',        'Conserver',  'Orientation panneaux PV'),
    ('in.battery',                   'Production',        'Conserver',  'Stockage batterie — flexibilité côté production'),
]

df_recap = pd.DataFrame(toutes_variables,
                         columns=['Variable', 'Catégorie', 'Décision', 'Justification'])

print('=== TABLEAU RÉCAPITULATIF ===\n')
conserver = df_recap[df_recap['Décision'] == 'Conserver']
supprimer = df_recap[df_recap['Décision'] == 'Supprimer']
sep       = df_recap[df_recap['Décision'] == 'Traiter séparément']

print(f'Variables à CONSERVER       : {len(conserver)}')
print(f'Variables à SUPPRIMER       : {len(supprimer)}')
print(f'Variables à traiter séparément (VE) : {len(sep)}')
print()

print('Variables MANQUANTES importantes (non dans ResStock metadata) :')
print('  ✗ Température extérieure horaire — disponible uniquement dans les fichiers TMY (.epw)')
print('  ✗ Humidité relative, irradiation, vent — idem')
print('  ✗ Horaires de présence des occupants — simulés en interne par EnergyPlus')
print('  ✗ Prix de l\'électricité / tarif heure creuse — à ajouter depuis RTE/NREL')
print('  ✗ Intensité carbone du réseau horaire — à ajouter depuis RTE eco2mix')
print()

# Camembert des catégories
cat_counts = df_recap['Catégorie'].value_counts()
fig, ax = plt.subplots(figsize=(10, 7))
colors_recap = plt.cm.Set3(np.linspace(0, 1, len(cat_counts)))
wedges, texts, autotexts = ax.pie(
    cat_counts.values, labels=cat_counts.index, colors=colors_recap,
    autopct='%1.0f%%', startangle=90, pctdistance=0.82
)
for at in autotexts: at.set_fontsize(9)
ax.set_title('Répartition des variables par catégorie métier', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'recap_categories_variables.png', dpi=150, bbox_inches='tight')
plt.show()

# Export tableau
df_recap.to_csv(ROOT / 'reports' / 'classification_variables_metier.csv', index=False, encoding='utf-8-sig')
print('\nTableau exporté → reports/classification_variables_metier.csv')

---
## 12. Schéma Hiérarchique Final

In [ ]:
schema = """
CONSOMMATION ÉNERGÉTIQUE RÉSIDENTIELLE
│
├── Chauffage          → out.electricity.heating.*
│     Pilotable ✓ (thermostat programmable)
│
├── Climatisation      → out.electricity.cooling.*
│     Pilotable ✓ (setpoint offset)
│
├── DHW (Eau Chaude)   → out.electricity.hot_water.*
│     Pilotable ✓ (heure de chauffe programmable)
│
├── Éclairage          → out.electricity.lighting_*  + ceiling_fan
│     Pilotable ~
│
├── Ventilation        → out.electricity.mech_vent
│     Pilotable ~
│
├── Usages Spécifiques
│   ├── Blanc (électroménager)
│   │     clothes_dryer + clothes_washer + dishwasher
│   │     + refrigerator + freezer + range_oven
│   │     Pilotable ~ (sèche-linge, lave-vaisselle horaires décalables)
│   │
│   └── Brun (électronique)
│         television + plug_loads
│         Non pilotable ✗
│
└── VE (traitement séparé)
      ev_charging  — Pilotable ✓✓ (recharge décalable)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

FACTEURS EXPLICATIFS
│
├── Occupants          in.occupants, in.usage_level, in.vacancy_status
│
├── Comportement       in.heating_setpoint*, in.cooling_setpoint*
│                      in.*_usage_level, in.natural_ventilation
│
├── Météo (proxys)     in.ashrae_iecc_climate_zone_2004
│                      in.building_america_climate_zone
│                      in.weather_file_latitude/longitude
│
├── Enveloppe thermique in.insulation_*, in.air_leakage_*
│                       in.windows, in.geometry_*, in.neighbors
│
├── HVAC               in.hvac_heating_type, in.hvac_cooling_type
│                      in.hvac_*_efficiency, in.heating_fuel
│                      in.water_heater_fuel, in.water_heater_efficiency
│
└── Socio-économique   in.income, in.federal_poverty_level
                       in.tenure, in.area_median_income
"""
print(schema)